# MARBERT Fine-tuning: Joint Learning versus Sentiment Only

**Status: not executed in the onboarding CPU environment.** Hugging Face model downloads were denied by the environment network policy. This notebook is ready for a GPU runtime with model access; it is not evidence of completed MARBERT training.

The shared MARBERT encoder feeds two classification heads. Joint loss is sentiment cross-entropy + sarcasm cross-entropy. The control uses sentiment loss alone. Both select their best epoch using validation sentiment macro-F1.

Recommended: a GPU runtime; batch size 8 and sequence length 128 are starting settings, not guaranteed memory requirements. Reduce batch size if necessary. Run the following installation cell in a fresh notebook runtime.

In [ ]:
from pathlib import Path
import os, subprocess

# Colab opens the notebook without checking out the rest of the repository.
if not Path("pyproject.toml").exists():
    if Path.cwd().name == "notebooks" and (Path.cwd().parent / "pyproject.toml").exists():
        os.chdir(Path.cwd().parent)
    else:
        checkout = Path("/content/NLP_Capstone") if Path("/content").exists() else Path.cwd() / "NLP_Capstone"
        if not checkout.exists():
            subprocess.run(["git", "clone", "https://github.com/iYm10/NLP_Capstone.git", str(checkout)], check=True)
        os.chdir(checkout)
assert Path("pyproject.toml").exists(), "Project checkout was not found."
print("Project directory:", Path.cwd())

In [ ]:
%pip install -e ".[train,dev]"

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
os.chdir(ROOT)
# Editable installs may not refresh an already-running notebook kernel.
import importlib
source_dir = str(ROOT / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
import maqsad
print("Project package:", maqsad.__file__)

import torch
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Use a GPU runtime for the complete MARBERT experiment."

## Model revision
Resolve and record an immutable model revision before both runs. Internet access to Hugging Face and its artifact hosts is required. The dataset downloader separately verifies immutable data hashes.

## Persistent checkpoint storage
In Google Colab, authorize the Google Drive mount when prompted. Training writes directly to `MyDrive/Maqsad/artifacts`, so completed checkpoints survive runtime deletion. A new run uses a separate timestamped directory. Outside Colab, outputs remain in the local artifacts directory. A disconnected runtime does not automatically resume an interrupted epoch.

In [ ]:
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
try:
    from google.colab import drive
except ImportError:
    OUTPUT_ROOT = ROOT / "artifacts" / RUN_ID
else:
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/Maqsad/artifacts") / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Checkpoints and reports will be saved to:", OUTPUT_ROOT)

In [ ]:
from huggingface_hub import model_info
REVISION = model_info("UBC-NLP/MARBERT").sha
print("MARBERT revision:", REVISION)

In [ ]:
for task in ["sentiment", "joint"]:
    subprocess.run([sys.executable, "-m", "maqsad.train", "--task", task, "--revision", REVISION,
        "--epochs", "3", "--batch-size", "8", "--seed", "42", "--output", str(OUTPUT_ROOT / f"marbert-{task}")], check=True)

In [ ]:
import pandas as pd
reports = {task: json.loads((OUTPUT_ROOT / f"marbert-{task}" / "metrics.json").read_text()) for task in ["sentiment", "joint"]}
display(pd.DataFrame({task: {"sentiment_macro_f1": report["test"]["sentiment_macro_f1"], "sentiment_accuracy": report["test"]["sentiment_accuracy"]} for task, report in reports.items()}))
print("Joint sarcasm F1:", reports["joint"]["test"]["sarcasm_f1"])

In [ ]:
# Restore the source path even if the live kernel has not loaded the editable install.
from pathlib import Path
import sys, importlib
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/NLP_Capstone")]
project_root = next((p for p in candidates if (p / "src/maqsad/inference.py").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Project source is missing. Run the checkout and installation cells first; do not rerun training.")
source_dir = str(project_root / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
from maqsad.inference import predict_transformer

if "OUTPUT_ROOT" not in globals():
    raise RuntimeError("Set OUTPUT_ROOT to your existing timestamped training folder in Drive. Do not create a new run or retrain.")
checkpoint = Path(OUTPUT_ROOT) / "marbert-joint"
if not (checkpoint / "metrics.json").is_file() or not (checkpoint / "weights.pt").is_file():
    raise FileNotFoundError(f"Completed joint checkpoint not found at {checkpoint}. Check the saved training path and training output.")
predict_transformer(["يا سلام على الخدمة، ساعتين وأنا أنتظر!"], checkpoint=checkpoint)

## Interpretation
A positive difference for a single seed is preliminary. Repeat both tasks with identical seeds 42, 43 and 44 and report mean and standard deviation. Do not tune on the test set. Copy only final aggregate metrics into reports; keep checkpoints and raw tweets out of Git.

## Export reports for GitHub
This small archive contains aggregate metrics and run metadata. It excludes large model weights, which remain in the checkpoint directory. Download the executed notebook separately using **File → Download → Download .ipynb**. The export does not push to GitHub or require a GitHub token.

In [ ]:
import shutil, platform
export_dir = OUTPUT_ROOT / "github-results"
export_dir.mkdir(exist_ok=True)
for task in ["sentiment", "joint"]:
    shutil.copy2(OUTPUT_ROOT / f"marbert-{task}" / "metrics.json", export_dir / f"marbert-{task}-metrics.json")
(export_dir / "run.json").write_text(json.dumps({"run_id": RUN_ID, "model_revision": REVISION,
    "python": platform.python_version(), "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0), "source_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()}, indent=2))
archive = shutil.make_archive(str(OUTPUT_ROOT / "maqsad-results"), "zip", export_dir)
print("Results archive:", archive)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(archive)